In [18]:
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [2]:
!pip install langchain_google_genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 23.8 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [21]:
class AgentState(TypedDict):
  messages:Annotated[Sequence[BaseMessage], add_messages]


In [12]:
!pip install -U langchain-google-genai

In [16]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ['GOOGLE_API_KEY']="AQ.Ab8RN6LkeG_6c254lu3Pz-U5NrzPHX4I-AkcpbMHxgGI-nLqtQ"
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [22]:
def assistant(state: AgentState):

    system_message = SystemMessage(
        content="You are a helpful study assistant. Remember the conversation and answer clearly."
    )

    response = model.invoke(
        [system_message] + list(state["messages"])
    )

    return {
        "messages": [response]
    }

In [25]:
graph=StateGraph(AgentState)
graph.add_node("assistant",assistant)
graph.add_edge(START,'assistant')
graph.add_edge('assistant',END)
#we created a memory
memory=MemorySaver()
app=graph.compile(checkpointer=memory)
config = {
    "configurable": {
        "thread_id": "student_1"
    }
}
result = app.invoke(
    {
        "messages": [
            HumanMessage(
                content="My name is Anushka and I am learning LangGraph."
            )
        ]
    },
    config=config
)

print("AI:", result["messages"][-1].content)

AI: Hi Anushka! It's great to meet you. LangGraph is a powerful tool.

How can I help you with LangGraph today? Do you have any specific questions, or are you looking for an overview, examples, or something else?
